# MiniProject1 — Text Analysis (hvo5, Hien Vo)

## 1. The question

**Can the frequency of the most common words in a book identify who wrote it — and how much of that signal is really *authorial style* rather than just *subject matter*?**

Word-frequency "fingerprints" are the classic answer to authorship attribution, but a naive comparison of all words is easily fooled: two books about rivers will share river words no matter who wrote them. So I am splitting the question in two:

1. **Within-author vs. between-author.** Are two books by the same author measurably more similar to each other than to a book by a different author?
2. **Style vs. topic.** Does that similarity survive when I throw away all the content words and keep only *function words* (the, of, would, which, her, upon...)? Function words are chosen unconsciously and are nearly independent of plot, so if the signal holds up there, it is a claim about style.
3. **Which words give an author away?** Are there specific words one author strongly prefers over another (README Assignment 1, question 3)?

## 2. Data sources

Plain-text books from **[Project Gutenberg](https://www.gutenberg.org/)**, three authors × two works each. Two authors are enough to answer the assignment; the third acts as a control so I can see whether "same author" distances are small in general or just small for one lucky pair.

| Author | Work | PG ID | Text URL |
| --- | --- | --- | --- |
| Mark Twain | The Adventures of Tom Sawyer | 74 | https://www.gutenberg.org/files/74/74-0.txt |
| Mark Twain | Adventures of Huckleberry Finn | 76 | https://www.gutenberg.org/files/76/76-0.txt |
| Charles Dickens | A Tale of Two Cities | 98 | https://www.gutenberg.org/files/98/98-0.txt |
| Charles Dickens | Great Expectations | 1400 | https://www.gutenberg.org/files/1400/1400-0.txt |
| Jane Austen | Pride and Prejudice | 1342 | https://www.gutenberg.org/files/1342/1342-0.txt |
| Jane Austen | Emma | 158 | https://www.gutenberg.org/files/158/158-0.txt |

Why these: all six are 19th-century English novels of comparable length (~110k–190k words), so I am not accidentally measuring century or genre. Twain writes American vernacular in first person, Dickens writes dense third-person English prose, and Austen writes free indirect discourse — three genuinely different styles, which makes it a fair test of whether the method can also spot the *small* differences between two books by the same person.

`pg4680.txt` (F. C. Adams, *Manuel Pereira*), already in this repo, is my smoke test: I develop the pipeline against it before downloading anything, and it doubles as a fourth author with a single work if I want an unknown-sample test.

## 3. Approach

1. **Acquire and cache.** Fetch each book once with `requests`, write it to a local `data/` directory, and read from disk afterwards so re-running the notebook does not hammer Gutenberg (and still works if the site blocks me, per the note in `MiniProject1.ipynb`).
2. **Strip boilerplate.** Keep only the text between the `*** START OF THIS PROJECT GUTENBERG EBOOK ***` and `*** END OF ...` markers, so the license, transcriber notes, and chapter tables of contents do not pollute the counts.
3. **Tokenize.** Lowercase, then split on a regex that keeps letters and internal apostrophes (`[a-z']+`) so `don't` stays one token and punctuation, digits, and em-dashes are dropped. Store counts in a `collections.Counter`.
4. **Normalize.** Books differ in length, so every comparison uses **relative frequency (occurrences per 10,000 tokens)**, never raw counts.
5. **Build two feature sets from the same tokens.**
   - *All words* — the naive fingerprint.
   - *Function words only* — NLTK's English stopword list plus auxiliaries and prepositions, with **proper nouns and character names removed** (`Huck`, `Pip`, `Elizabeth` are the single biggest give-away of a *book*, not an *author*).
6. **Compare, four ways.**
   - **Rank–frequency (Zipf) plots** on log–log axes, and side-by-side bar charts of the top ~30 words for each pair — the visual, sanity-check layer.
   - **Spearman rank correlation** of the top 200 shared words: do the two books order their vocabulary the same way?
   - **Cosine similarity** on the function-word frequency vectors.
   - **Burrows's Delta** — z-score each of the top *N* function words across the corpus, then take the mean absolute z-difference between two books. This is the standard stylometry distance, and it gives me one number per book pair. I will report all 15 pairs as a distance matrix and check the central prediction: **every within-author distance should be smaller than every between-author distance.** Where that fails is the interesting part of the writeup.
7. **Find the give-away words.** For each author pair, rank words by **log-likelihood (keyness)** and by frequency ratio, requiring a minimum count so rare words cannot top the list. Expected outcome: Twain-vs-Dickens is dominated by dialect (`ain't`, `warn't`, `reckon`) while Dickens-vs-Austen turns on discourse habits (`upon`, `indeed`, `very`).
8. **Extra credit — is the difference statistically real?** Model a chosen word (`would`) as binomial: each token is a trial with probability *p* = N(`would`)/N(tokens). Estimate *p* per book with a confidence interval, then use a **two-proportion z-test** (and chi-square as a cross-check) to ask whether *p* differs significantly *between* authors while staying stable *within* an author. This distinguishes "these numbers look different" from "these numbers are different."

## 4. Problems I expect to hit

- **Topic leakage.** Character and place names will dominate any all-words comparison. The function-word run is the fix, and comparing the two runs is itself a result worth reporting.
- **Dialect.** Huckleberry Finn's phonetic spelling explodes the vocabulary and will make Twain's two books look less alike than they are — a real limitation of word-level features.
- **Length and sampling.** *Emma* is much longer than *Tom Sawyer*. Beyond per-10k normalization, I plan to chunk each book into equal-size slices (~10k words) so I can see *variance within a single book* and know whether a between-book difference is bigger than the noise inside one book. Without this the significance tests are over-confident, since word choices are not really independent trials.
- **Encoding.** Gutenberg files carry BOMs and smart quotes; I will read as UTF-8 with `errors='replace'` rather than the ASCII fallback used in the example notebook.
- **Multiple comparisons.** Scanning thousands of words for "significant" differences will produce false positives, so the keyness lists get a frequency floor and I will note the correction rather than quietly reporting the top hit.

## 5. Deliverables

A single reproducible notebook containing the download/clean pipeline, the Zipf and top-word plots, the 6×6 Delta distance matrix for both feature sets, the distinctive-word tables per author pair, and the binomial test for `would` — closing with what the results say about the style-vs-topic question and what I would try next (character *n*-grams, or attributing an unseen text such as `pg4680.txt`).

**Timeline:** initial description committed by Sep 3; pipeline and first plots by the weekend; at least three commits and complete analysis plus slides by Sep 10.